# Unit 03: Matrix Multiplication and PyTorch Intuition

This unit is about intuition, not implementing matrix multiplication from scratch. Focus on what `A @ B` means, why order matters, and how PyTorch shapes behave.

## Setup

Import PyTorch before starting.

API you may need:

```python
import torch
torch.tensor([...], dtype=torch.float32)
A.shape
A @ B
torch.allclose(x, y)
```

In [7]:
import torch

## The Big Idea: Composition

A matrix can represent a transformation. Matrix multiplication composes transformations.

If `A` and `B` are matrices, then:

```text
(A @ B) @ v = A @ (B @ v)
```

Read this right-to-left: apply `B` to `v` first, then apply `A`.

That is why order matters. `A @ B` means "B then A". `B @ A` means "A then B". Those usually describe different transformations.

## Shape Rule

For matrix multiplication, the inner dimensions must match:

```text
(m, k) @ (k, n) -> (m, n)
```

Examples:

```text
(2, 2) @ (2, 2) -> (2, 2)
(5, 3) @ (3, 2) -> (5, 2)
(3, 4) @ (4,)   -> (3,)
```

Before every `@`, say the shapes out loud.

## Example 1: AB vs BA

First compute `AB` and `BA` by hand for these matrices:

```text
A = [[1, 2],
     [0, 1]]

B = [[1, 0],
     [3, 1]]
```

Then use PyTorch to verify. Do not implement matrix multiplication manually. Just use `@`.

### Your hand work for Example 1

Write your hand-computed `AB`, `BA`, and one sentence explaining why they differ.

(paper)

### PyTorch verification for Example 1

Create `A` and `B`, print their shapes, compute `A @ B`, compute `B @ A`, and check whether they are equal.

API you may need:

```python
A = torch.tensor(..., dtype=torch.float32)
B = torch.tensor(..., dtype=torch.float32)
AB = A @ B
BA = B @ A
torch.allclose(AB, BA)
```

In [9]:
A = torch.tensor([[1, 2.], [0, 1.]])
B = torch.tensor([[1, 0.], [3, 1.]])
AB = A @ B
print(AB)

tensor([[7., 2.],
        [3., 1.]])


In [11]:
BA = B @ A
print(BA)

tensor([[1., 2.],
        [3., 7.]])


## Composition Check with a Vector

Pick a vector `v = (2, 1)`. Verify the composition identity:

```text
(A @ B) @ v == A @ (B @ v)
```

Then compare it to:

```text
(B @ A) @ v == B @ (A @ v)
```

Notice that both identities are true, but they represent different orders of transformation.

API you may need:

```python
v = torch.tensor([...], dtype=torch.float32)
(A @ B) @ v
A @ (B @ v)
torch.allclose(left, right)
```

In [12]:
v = torch.tensor([2, 1.])
(A @ B) @ v == A @ (B @ v)

tensor([True, True])

In [13]:
(B @ A) @ v == B @ (A @ v)

tensor([True, True])

## Column View of AB

Another useful way to read `A @ B`:

```text
Each column of A @ B is A applied to the matching column of B.
```

So if `B` has columns `b1` and `b2`, then:

```text
A @ B = [A @ b1, A @ b2]
```

This links back to matrices as linear maps: the columns tell you where basis directions land after the composed transformation.

### Verify the column view

Compute `AB = A @ B`. Then pull out the first and second columns of `B`, apply `A` to each one, and stack the results back together. Confirm that this reconstructed matrix equals `AB`.

API you may need:

```python
B[:, 0]
B[:, 1]
A @ b1
torch.stack([col1, col2], dim=1)
torch.allclose(reconstructed, AB)
```

In [14]:
AB = A @ B
print(AB)

tensor([[7., 2.],
        [3., 1.]])


In [16]:
# Take each column of b
b1 = B[:, 0]
b2 = B[:, 1]
print(b1)
print(b2)

tensor([1., 3.])
tensor([0., 1.])


In [17]:
# Intermediate step taking each b column, applying the A transformation to the b vector
col1 = A @ b1
print(col1)

col2 = A @ b2
print(col2)

tensor([7., 3.])
tensor([2., 1.])


In [18]:
column_view_answer = torch.stack([col1, col2], dim=1)
print(column_view_answer)

tensor([[7., 2.],
        [3., 1.]])


In [19]:
torch.allclose(column_view_answer, AB)

True

## Example 2: Another AB Practice

Do one more by-hand multiplication, then verify in PyTorch:

```text
C = [[2, 0],
     [1, 3]]

D = [[0, 1],
     [4, 2]]
```

Tasks:

1. Compute `C @ D` by hand.
2. Compute `D @ C` by hand.
3. Verify both in PyTorch.
4. Check whether `C @ D` equals `D @ C`.
5. Apply both composed transformations to `w = (1, 2)` and compare the outputs.

### Your hand work for Example 2

Write your hand-computed `C @ D`, `D @ C`, and what happened to `w = (1, 2)`.

(paper)

### PyTorch verification for Example 2

API you may need:

```python
C = torch.tensor(..., dtype=torch.float32)
D = torch.tensor(..., dtype=torch.float32)
w = torch.tensor(..., dtype=torch.float32)
CD = C @ D
DC = D @ C
CD @ w
DC @ w
torch.allclose(CD, DC)
```

In [20]:
# written where the columns position of each row is the image of the basis vector in this matrix
# each column is where this matrix sends that basis (ie e1, e2)
C = torch.tensor([[2, 0.], [1, 3.]])
D = torch.tensor([[0, 1.], [4, 2.]])
print(C@D)

tensor([[ 0.,  2.],
        [12.,  7.]])


In [21]:
print(D@C)

tensor([[ 1.,  3.],
        [10.,  6.]])


In [22]:
C@D == D@C

tensor([[False, False],
        [False, False]])

In [24]:
w = torch.tensor([1, 2.])
CD = C@D
print(CD @ w)

tensor([ 4., 26.])


In [26]:
DC = D@C
print(DC @ w)

tensor([ 7., 22.])


## Neural Network Link

Ignoring bias and nonlinearities, stacked neural network layers are composed matrix transformations.

```text
h = x @ W1
y = h @ W2

So y = (x @ W1) @ W2
```

The order and shapes matter. If `x` is a batch with shape `(batch_size, input_features)`, then:

```text
X:  (batch_size, input_features)
W1: (input_features, hidden_features)
H:  (batch_size, hidden_features)
W2: (hidden_features, output_features)
Y:  (batch_size, output_features)
```

### Shape practice: two linear layers without bias

Create random tensors with these shapes and verify the output shapes:

```text
X:  (4, 3)
W1: (3, 5)
W2: (5, 2)
H = X @ W1
Y = H @ W2
```

Then compare `Y` to `X @ (W1 @ W2)`. They should match because matrix multiplication is associative when the shapes line up.

API you may need:

```python
torch.randn(rows, cols)
X @ W1
H @ W2
W1 @ W2
torch.allclose(Y, X @ (W1 @ W2))
```

In [29]:
X = torch.randn(4, 3) # a batch of 4, 3 dimensional input vectors
W1 = torch.randn(3, 5)
W2 = torch.randn(5, 2)
print(X)

tensor([[ 0.4672,  1.4522,  0.8033],
        [-0.1126, -0.3979, -0.5649],
        [ 0.6765, -1.3837, -0.4356],
        [-0.0039,  0.7074,  0.0249]])


In [30]:
H = X @ W1
# 4 samples from X, projected from 3 dimensions to 5 by W1
print(H)

tensor([[ 1.1814, -0.6427, -1.3617, -3.1285, -1.8597],
        [-0.5989,  0.4500,  0.3473,  0.7274,  0.6679],
        [-1.1249,  0.1809,  0.7698,  3.8008,  1.2521],
        [ 0.3403,  0.0150, -0.5703, -1.8136, -0.6573]])


In [31]:
Y = H @ W2
# 4 vectors from H layer projected from 5 dimensions to 2
print(Y)

tensor([[ 5.0211,  5.8554],
        [-1.5213, -2.0052],
        [-4.1001, -5.8293],
        [ 2.1724,  2.5645]])
